# Aggregate CSV Files by Timestamp

This notebook aggregates 4 CSV files (AQI, Indoor, Outdoor, Weather) by timestamp and removes null values.

In [28]:
import pandas as pd
import numpy as np
import os

# Set data directory
data_dir = '../models/data/'
print(f"Data directory: {data_dir}")
print(f"Files in directory: {os.listdir(data_dir)}")

Data directory: ../models/data/
Files in directory: ['indoor.csv', 'outdoor.csv', 'weather_data.csv', 'aqi_data.csv']


In [29]:
# Load all 4 CSV files
aqi_df = pd.read_csv(os.path.join(data_dir, 'aqi_data.csv'))
indoor_df = pd.read_csv(os.path.join(data_dir, 'indoor.csv'))
outdoor_df = pd.read_csv(os.path.join(data_dir, 'outdoor.csv'))
weather_df = pd.read_csv(os.path.join(data_dir, 'weather_data.csv'))

print("AQI Data shape:", aqi_df.shape)
print("AQI columns:", aqi_df.columns.tolist())
print("\nIndoor Data shape:", indoor_df.shape)
print("Indoor columns:", indoor_df.columns.tolist())
print("\nOutdoor Data shape:", outdoor_df.shape)
print("Outdoor columns:", outdoor_df.columns.tolist())
print("\nWeather Data shape:", weather_df.shape)
print("Weather columns:", weather_df.columns.tolist())

AQI Data shape: (69, 2)
AQI columns: ['ts', 'aqi']

Indoor Data shape: (529, 5)
Indoor columns: ['ts', 'temp_dht', 'pm1', 'pm25', 'pm10']

Outdoor Data shape: (644, 5)
Outdoor columns: ['ts', 'temp_dht', 'pm1', 'pm25', 'pm10']

Weather Data shape: (117, 5)
Weather columns: ['ts', 'humid', 'rainfall', 'temp', 'windspeed']


In [30]:
# Identify timestamp columns (common names: timestamp, time, datetime, date_time)
def find_timestamp_column(df):
    """Find timestamp column in dataframe"""
    timestamp_candidates = ['timestamp', 'time', 'datetime', 'date_time', 'Date', 'Time']
    for col in timestamp_candidates:
        if col in df.columns:
            return col
    # If none found, return first column
    return df.columns[0]

# Get timestamp column names for each dataframe
ts_aqi = find_timestamp_column(aqi_df)
ts_indoor = find_timestamp_column(indoor_df)
ts_outdoor = find_timestamp_column(outdoor_df)
ts_weather = find_timestamp_column(weather_df)

print(f"AQI timestamp column: {ts_aqi}")
print(f"Indoor timestamp column: {ts_indoor}")
print(f"Outdoor timestamp column: {ts_outdoor}")
print(f"Weather timestamp column: {ts_weather}")

AQI timestamp column: ts
Indoor timestamp column: ts
Outdoor timestamp column: ts
Weather timestamp column: ts


In [31]:
# Convert timestamp columns to datetime and check intervals
aqi_df['ts'] = pd.to_datetime(aqi_df['ts'])
indoor_df['ts'] = pd.to_datetime(indoor_df['ts'])
outdoor_df['ts'] = pd.to_datetime(outdoor_df['ts'])
weather_df['ts'] = pd.to_datetime(weather_df['ts'])

# Check timestamp intervals
print("Timestamp intervals (minutes):")
print(f"AQI: {(aqi_df['ts'].diff().dt.total_seconds() / 60).mode().values[0]:.0f} min" if len(aqi_df) > 1 else "AQI: N/A")
print(f"Indoor: {(indoor_df['ts'].diff().dt.total_seconds() / 60).mode().values[0]:.0f} min" if len(indoor_df) > 1 else "Indoor: N/A")
print(f"Outdoor: {(outdoor_df['ts'].diff().dt.total_seconds() / 60).mode().values[0]:.0f} min" if len(outdoor_df) > 1 else "Outdoor: N/A")
print(f"Weather: {(weather_df['ts'].diff().dt.total_seconds() / 60).mode().values[0]:.0f} min" if len(weather_df) > 1 else "Weather: N/A")

Timestamp intervals (minutes):
AQI: 10 min
Indoor: 10 min
Outdoor: 10 min
Weather: 10 min


In [32]:
# Resample AQI and Weather data to 10-minute intervals to match Indoor/Outdoor
# Set timestamp as index for resampling
aqi_df_resampled = aqi_df.set_index('ts').resample('10min').ffill().reset_index()
weather_df_resampled = weather_df.set_index('ts').resample('10min').ffill().reset_index()

print(f"AQI resampled from {len(aqi_df)} to {len(aqi_df_resampled)} rows")
print(f"Weather resampled from {len(weather_df)} to {len(weather_df_resampled)} rows")
print(f"\nNew timestamp intervals (10 minutes):")
print(f"AQI: {(aqi_df_resampled['ts'].diff().dt.total_seconds() / 60).mode().values[0]:.0f} min")
print(f"Weather: {(weather_df_resampled['ts'].diff().dt.total_seconds() / 60).mode().values[0]:.0f} min")

# Update original dataframes with resampled versions
aqi_df = aqi_df_resampled
weather_df = weather_df_resampled

AQI resampled from 69 to 69 rows
Weather resampled from 117 to 131 rows

New timestamp intervals (10 minutes):
AQI: 10 min
Weather: 10 min


In [33]:
# Merge all dataframes by timestamp
# Start with one dataframe and merge others to it
aggregated_df = aqi_df.copy()
aggregated_df = aggregated_df.rename(columns={ts_aqi: 'timestamp'})

# Merge with indoor data
indoor_df_copy = indoor_df.copy()
indoor_df_copy = indoor_df_copy.rename(columns={ts_indoor: 'timestamp'})
aggregated_df = aggregated_df.merge(indoor_df_copy, on='timestamp', how='outer')

# Merge with outdoor data
outdoor_df_copy = outdoor_df.copy()
outdoor_df_copy = outdoor_df_copy.rename(columns={ts_outdoor: 'timestamp'})
aggregated_df = aggregated_df.merge(outdoor_df_copy, on='timestamp', how='outer')

# Merge with weather data
weather_df_copy = weather_df.copy()
weather_df_copy = weather_df_copy.rename(columns={ts_weather: 'timestamp'})
aggregated_df = aggregated_df.merge(weather_df_copy, on='timestamp', how='outer')

print(f"Aggregated DataFrame shape: {aggregated_df.shape}")
print(f"\nColumns: {aggregated_df.columns.tolist()}")
print(f"\nNull values per column BEFORE cleaning:")
print(aggregated_df.isnull().sum())

Aggregated DataFrame shape: (821, 14)

Columns: ['timestamp', 'aqi', 'temp_dht_x', 'pm1_x', 'pm25_x', 'pm10_x', 'temp_dht_y', 'pm1_y', 'pm25_y', 'pm10_y', 'humid', 'rainfall', 'temp', 'windspeed']

Null values per column BEFORE cleaning:
timestamp       0
aqi           753
temp_dht_x    292
pm1_x         292
pm25_x        292
pm10_x        292
temp_dht_y    177
pm1_y         177
pm25_y        177
pm10_y        177
humid         691
rainfall      691
temp          691
windspeed     691
dtype: int64


In [34]:
# Handle null values - choose a strategy based on your needs
print(f"Rows before null removal: {len(aggregated_df)}")
print("\nNull percentage per column:")
print((aggregated_df.isnull().sum() / len(aggregated_df) * 100).round(2))

# Strategy 1: Forward fill then drop remaining nulls (most practical)
aggregated_df_clean = aggregated_df.copy()
aggregated_df_clean = aggregated_df_clean.sort_values('timestamp')
aggregated_df_clean = aggregated_df_clean.ffill().bfill()

# Strategy 2 (Alternative if you want stricter): Drop rows where ANY value is null
# aggregated_df_clean = aggregated_df.dropna()

# Strategy 3 (Alternative): Drop rows where specific columns are null (customize as needed)
# aggregated_df_clean = aggregated_df.dropna(subset=['aqi', 'pm1_x', 'pm25_x', 'pm10_x'])

print(f"\nRows after null removal: {len(aggregated_df_clean)}")
print(f"\nNull values per column AFTER cleaning:")
print(aggregated_df_clean.isnull().sum())
print(f"\nTotal null values in cleaned dataframe: {aggregated_df_clean.isnull().sum().sum()}")

Rows before null removal: 821

Null percentage per column:
timestamp      0.00
aqi           91.72
temp_dht_x    35.57
pm1_x         35.57
pm25_x        35.57
pm10_x        35.57
temp_dht_y    21.56
pm1_y         21.56
pm25_y        21.56
pm10_y        21.56
humid         84.17
rainfall      84.17
temp          84.17
windspeed     84.17
dtype: float64

Rows after null removal: 821

Null values per column AFTER cleaning:
timestamp     0
aqi           0
temp_dht_x    0
pm1_x         0
pm25_x        0
pm10_x        0
temp_dht_y    0
pm1_y         0
pm25_y        0
pm10_y        0
humid         0
rainfall      0
temp          0
windspeed     0
dtype: int64

Total null values in cleaned dataframe: 0


In [35]:
# Rename PM columns to be more descriptive
column_mapping = {
    'pm1_x': 'pm1_indoor',
    'pm25_x': 'pm25_indoor',
    'pm10_x': 'pm10_indoor',
    'pm1_y': 'pm1_outdoor',
    'pm25_y': 'pm25_outdoor',
    'pm10_y': 'pm10_outdoor',
    'temp_dht_x': 'temp_indoor',
    'temp_dht_y': 'temp_outdoor'
}

aggregated_df_clean = aggregated_df_clean.rename(columns=column_mapping)

print("Column names updated:")
print(aggregated_df_clean.columns.tolist())

Column names updated:
['timestamp', 'aqi', 'temp_indoor', 'pm1_indoor', 'pm25_indoor', 'pm10_indoor', 'temp_outdoor', 'pm1_outdoor', 'pm25_outdoor', 'pm10_outdoor', 'humid', 'rainfall', 'temp', 'windspeed']


In [36]:
# Analyze data quality - check how much was forward-filled
print("=== DATA QUALITY ANALYSIS ===\n")

print("ORIGINAL DATA SIZES:")
print(f"  AQI: {len(aqi_df_resampled)} rows (after resampling)")
print(f"  Weather: {len(weather_df_resampled)} rows (after resampling)")
print(f"  Indoor: {len(indoor_df)} rows")
print(f"  Outdoor: {len(outdoor_df)} rows")

print("\nAGGREGATED (before null fill):")
print(f"  Total rows: {len(aggregated_df)}")

print("\nAGGREGATED (after forward/backward fill):")
print(f"  Total rows: {len(aggregated_df_clean)}")

# Count how many values were filled (not original measurements)
aqi_filled = aggregated_df['aqi'].isna().sum()
weather_filled = (aggregated_df['humid'].isna().sum() + aggregated_df['rainfall'].isna().sum() + 
                   aggregated_df['temp'].isna().sum() + aggregated_df['windspeed'].isna().sum()) / 4

print(f"\nROWS WITH MISSING ORIGINAL DATA (before fill):")
print(f"  AQI missing: {aqi_filled} rows ({aqi_filled/len(aggregated_df)*100:.1f}%)")
print(f"  Weather missing: ~{weather_filled:.0f} rows (~{weather_filled/len(aggregated_df)*100:.1f}%)")

print("\n⚠️  WARNING: Most AQI and Weather data is FORWARD-FILLED, not actual measurements!")
print("   This will bias your model training.\n")

print("RECOMMENDATIONS:")
print("  1. Use INNER JOIN instead (keep only rows with all sensors)")
print("  2. Use only Indoor/Outdoor data as main features")
print("  3. Drop rows where AQI/Weather are filled values")

=== DATA QUALITY ANALYSIS ===

ORIGINAL DATA SIZES:
  AQI: 69 rows (after resampling)
  Weather: 131 rows (after resampling)
  Indoor: 529 rows
  Outdoor: 644 rows

AGGREGATED (before null fill):
  Total rows: 821

AGGREGATED (after forward/backward fill):
  Total rows: 821

ROWS WITH MISSING ORIGINAL DATA (before fill):
  AQI missing: 753 rows (91.7%)
  Weather missing: ~691 rows (~84.2%)

⚠️  WARNING: Most AQI and Weather data is FORWARD-FILLED, not actual measurements!
   This will bias your model training.

RECOMMENDATIONS:
  1. Use INNER JOIN instead (keep only rows with all sensors)
  2. Use only Indoor/Outdoor data as main features
  3. Drop rows where AQI/Weather are filled values


In [37]:
# ALTERNATIVE: Create aggregated data with INNER JOIN (only real measurements, no forward-fill)
print("\n=== CREATING ALTERNATIVE WITH INNER JOIN ===\n")

# Start fresh and use INNER JOIN instead of OUTER + fill
aqi_inner = aqi_df_resampled.copy()
aqi_inner = aqi_inner.rename(columns={'ts': 'timestamp'})

indoor_inner = indoor_df_copy.copy()
indoor_inner = indoor_inner.rename(columns={'ts': 'timestamp'})

outdoor_inner = outdoor_df_copy.copy()
outdoor_inner = outdoor_inner.rename(columns={'ts': 'timestamp'})

weather_inner = weather_df_resampled.copy()
weather_inner = weather_inner.rename(columns={'ts': 'timestamp'})

# Use INNER JOIN - keeps only rows where all 4 datasets have data
aggregated_inner = aqi_inner.merge(indoor_inner, on='timestamp', how='inner')
aggregated_inner = aggregated_inner.merge(outdoor_inner, on='timestamp', how='inner')
aggregated_inner = aggregated_inner.merge(weather_inner, on='timestamp', how='inner')

# Rename columns
aggregated_inner = aggregated_inner.rename(columns=column_mapping)

print(f"INNER JOIN result: {len(aggregated_inner)} rows (only actual simultaneous measurements)")
print(f"Data completeness: {len(aggregated_inner)/len(aggregated_df_clean)*100:.1f}% of outer join result\n")

if len(aggregated_inner) > 0:
    print(f"Timestamp range: {aggregated_inner['timestamp'].min()} to {aggregated_inner['timestamp'].max()}")
    print(aggregated_inner.head())


=== CREATING ALTERNATIVE WITH INNER JOIN ===

INNER JOIN result: 0 rows (only actual simultaneous measurements)
Data completeness: 0.0% of outer join result



In [ ]:
# REBUILD: Use ONLY Indoor + Outdoor data (real measurements, no made-up data)
print("=== CLEAN AGGREGATION (No Synthetic Data) ===\n")

# Reload just indoor and outdoor
indoor_clean = indoor_df_copy.copy()
indoor_clean = indoor_clean.rename(columns={'ts': 'timestamp'})
outdoor_clean = outdoor_df_copy.copy()
outdoor_clean = outdoor_clean.rename(columns={'ts': 'timestamp'})

# INNER JOIN on timestamp - only keeps rows where BOTH indoor AND outdoor measured
aggregated_clean = indoor_clean.merge(outdoor_clean, on='timestamp', how='inner')
aggregated_clean = aggregated_clean.rename(columns=column_mapping)

print(f"Indoor records: {len(indoor_clean)}")
print(f"Outdoor records: {len(outdoor_clean)}")
print(f"INNER JOIN (both present): {len(aggregated_clean)} rows")
print(f"\nTimestamp range: {aggregated_clean['timestamp'].min()} to {aggregated_clean['timestamp'].max()}")

# Check null values
print(f"\nNull values: {aggregated_clean.isnull().sum().sum()} (should be 0)")
print(f"\nColumns: {aggregated_clean.columns.tolist()}")

# Save this clean version
clean_output_path = os.path.join(data_dir, 'aggregated_data_clean.csv')
aggregated_clean.to_csv(clean_output_path, index=False)
print(f"\n✓ Saved clean data (Indoor + Outdoor only) to: aggregated_data_clean.csv")

In [38]:
# Sort by timestamp and save
aggregated_df_clean = aggregated_df_clean.sort_values('timestamp').reset_index(drop=True)

# Save to CSV
output_path = os.path.join(data_dir, 'aggregated_data.csv')
aggregated_df_clean.to_csv(output_path, index=False)

print(f"✓ Aggregated data saved to: {output_path}")
print(f"\nFinal dataset statistics:")
print(f"Shape: {aggregated_df_clean.shape}")
print(f"Timestamp range: {aggregated_df_clean['timestamp'].min()} to {aggregated_df_clean['timestamp'].max()}")
print(f"\nFirst few rows:")
aggregated_df_clean.head()

✓ Aggregated data saved to: ../models/data/aggregated_data.csv

Final dataset statistics:
Shape: (821, 14)
Timestamp range: 2026-03-26 21:40:00 to 2026-04-01 21:00:00

First few rows:


,timestamp,aqi,temp_indoor,pm1_indoor,pm25_indoor,pm10_indoor,temp_outdoor,pm1_outdoor,pm25_outdoor,pm10_outdoor,humid,rainfall,temp,windspeed
0,2026-03-26 21:40:00,77.0,28.0,6.5,7.0,7.3,2.636364,13.727273,18.0,19.272727,75.0,0.0,31.2,5.6
1,2026-03-26 21:50:00,77.0,28.0,6.5,7.0,7.3,2.636364,13.727273,18.0,19.272727,75.0,0.0,31.2,5.6
2,2026-03-26 22:00:00,77.0,28.0,6.5,7.0,7.3,2.636364,13.727273,18.0,19.272727,75.0,0.0,31.2,5.6
3,2026-03-26 22:10:00,77.0,28.0,6.5,7.0,7.3,2.636364,13.727273,18.0,19.272727,75.0,0.0,31.2,5.6
4,2026-03-26 22:20:00,77.0,28.0,6.5,7.0,7.3,2.636364,13.727273,18.0,19.272727,80.0,0.0,29.6,9.3
